In [ ]:
#  __      __  _____    ___________________ 
# /  \    /  \/  _  \  /   _____/\______   \
# \   \/\/   /  /_\  \ \_____  \  |     ___/
#  \        /    |    \/        \ |    |    
#   \__/\  /\____|__  /_______  / |____|    
#       \/         \/        \/         


# GET STUNG

# LFQ DIA Peptide-Level Analysis

In [ ]:
# Params
_PATH_ = "~/PATH/"
INPUT_PATH = f"{_PATH_}lfq.dia.peptides.csv"                   
METADATA_PATH = f"{_PATH_}sample_metadata.csv"                 # Must contain 'sample_id', 'group', 'label'
OUTPUT_FILTERED = f"{_PATH_}filtered_peptide_matrix.csv"
OUTPUT_AGGREGATED = f"{_PATH_}aggregated_protein_matrix.csv"
OUTPUT_RLE = f"{_PATH_}RLE.png"
OUTPUT_IMPUTE = f"{_PATH_}Post-imputation_histogram.png"
OUTPUT_PCA = f"{_PATH_}PCA.svg"
OUTPUT_UMAP2D = f"{_PATH_}UMAP-2D.svg"
OUTPUT_UMAP3D = f"{_PATH_}UMAP-3D.svg"
OUTPUT_SINGLE_PEPTIDE = f"{_PATH_}single_peptide.csv"


MIN_COMPLETENESS = 0.5  # Intragroup completeness threshold
CV_THRESHOLD = 3.5      # Intragroup CV upper limit
TOP_N = 3               # Top-N peptides per protein for aggregation
MIN_INTENSITY_Q = 0.10  # Minimum acceptable median intensity (percentile)
PSEUDOCOUNT = 1e-5      # For log-transform and imputation
N_GROUP = 2             # Number of groups
QUANTILE = 0.10         # Quantile to sample skewnormal imputation
SHIFT = 2.00            # Number of std deviations to shift the gaussian center to the left
WIDTH = 0.33            # Std deviation multiplier (more of less variance)
MIN_ALLOWED = None      # ABS minimum for imputed values
SEED = 42               # Raondom seed for reproducibility
LEGEND_LAB = 'Top Legend'
LEGEND_LAB1 = 'Label 1'
LEGEND_LAB2 = 'Label 2'

In [ ]:
# Load Packages
import pandas as pd
import numpy as np

from scipy.stats import median_abs_deviation, skewnorm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
    train_test_split,
    cross_val_score,
    cross_val_predict)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    VotingClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    BaggingClassifier,
    StackingClassifier)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_curve,
    roc_auc_score,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score,
    ConfusionMatrixDisplay)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, NeighborhoodComponentsAnalysis
from sklearn.feature_selection import RFE, RFECV, SelectKBest, f_classif
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.datasets import make_classification

from itertools import combinations

from umap import UMAP
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

import warnings
warnings.filterwarnings("ignore", message="n_jobs value 1 overridden to 1", category=UserWarning)

## Normalization Method

In [ ]:
# ---------------------- Method 1: Median Normalization (robust median centering) -------------------
def median_normalize(df):
    df_norm = df.copy()
    centers = df_norm.median(skipna=True)
    return df_norm.subtract(centers, axis=1)

# ---------------------- Method 2: TMM-like Scaling ----------------------------------
def tmm_normalize(df):
    df = df.copy()
    ref_sample = df.median(axis=1)  # pseudo-reference sample

    scaling_factors = []
    for col in df.columns:
        ratios = df[col] - ref_sample
        trimmed = ratios[(ratios > np.percentile(ratios, 5)) & (ratios < np.percentile(ratios, 95))]
        scaling_factors.append(trimmed.median())

    scale_series = pd.Series(scaling_factors, index=df.columns)
    return df - scale_series  # subtract log-scale factor per sample


#---------------------- Method 3: Variance Stabilizing Normalization -------------------
def vsn_normalize(df):
    df_trans = df.copy().apply(lambda x: 2**x)
    offset = np.median(df.values[np.isfinite(df.values)])

    # Apply transformation: log2(a + sqrt(x + offset))
    vsn_func = lambda x: np.log2(1.0 + np.sqrt(x + offset))
    df_vsn = df_trans.applymap(vsn_func)

    df_vsn = (df_vsn - df_vsn.mean()) / df_vsn.std()

    return df_vsn

#---------------------- Method 4: RobNorm-Like with MAD Scaling -----------------------
# Function for robust normalization
def robMAD_normalize(df_log2, preserve_index=True):
    normalized_df = df_log2.copy()
    for col in df_log2.columns:
        values = df_log2[col]
        center = np.median(values.dropna())
        scale = median_abs_deviation(values.dropna())
        # Avoid division by zero
        scale = scale if scale != 0 else 1.0
        normalized = (values - center) / scale

        normalized_df[col] = normalized

    if preserve_index:
        normalized_df.index = df_log2.index

    return normalized_d
    

## Imputation Method

In [ ]:
# ---------------------- Method 1: Left-Tail Skewnorm -------------------
def left_tail_skewnorm_impute(df, quantile=QUANTILE, seed=SEED):
    np.random.seed(seed)
    imputed_df = df.copy()

    for col in imputed_df.columns:
        observed = imputed_df[col].dropna()
        
        # Fit skew-normal: returns shape (a), location (loc), scale (scale)
        s, loc, scale = skewnorm.fit(observed)

        # Sample missing values from the left tail
        missing_mask = imputed_df[col].isna()
        n_missing = missing_mask.sum()
        if n_missing == 0:
            continue
        
        # Define upper bound in left tail: e.g., 10th percentile in the fitted skewnorm
        left_tail_max = skewnorm.ppf(quantile, s, loc, scale)
        # Impute from skewnorm, truncated to only accept samples <= left_tail_max
        imputed_vals = []
        while len(imputed_vals) < n_missing:
            sample = skewnorm.rvs(s, loc, scale, size=(n_missing - len(imputed_vals)))
            tail_samples = sample[sample <= left_tail_max]
            imputed_vals.extend(tail_samples)
        imputed_vals = np.array(imputed_vals)[:n_missing]
        imputed_df.loc[missing_mask, col] = imputed_vals

    return imputed_df

# ---------------------- Method 2: Left-Tail Gaussian -------------------
def left_shifted_gaussian_impute(df, shift=SHIFT, width=WIDTH, min_allowed=MIN_ALLOWED, seed=SEED):
    np.random.seed(seed)
    imputed_df = df.copy()

    for col in imputed_df.columns:
        col_data = imputed_df[col]
        missing_mask = col_data.isna()
        if missing_mask.sum() == 0:
            continue
        
        observed_vals = col_data[~missing_mask]
        mean_obs = observed_vals.mean()
        std_obs = observed_vals.std()

        low_gaussian = np.random.normal(loc=mean_obs - shift * std_obs,
                                        scale=width * std_obs,
                                        size=missing_mask.sum())

        if min_allowed is not None:
            low_gaussian = np.clip(low_gaussian, min_allowed, None)

        # Impute
        imputed_df.loc[missing_mask, col] = low_gaussian

    return imputed_df


## Aggregation Method

In [ ]:
# ------------------------ Method 1: Aggregate by Top-N most intense peptides (mean) -----------------
def aggregate_top_n(df, topn=TOP_N, remove_single_peptide=True, single_peptide_csv=None):
    pep_counts = df.index.get_level_values('Accession').value_counts()

    def topn_agg(protein_df):
        peptide_medians = protein_df.median(axis=1)
        top_peptides = peptide_medians.nlargest(topn).index
        return protein_df.loc[top_peptides].mean(axis=0)

    aggdf = (
        df
        .groupby(level='Accession', group_keys=False)
        .apply(topn_agg)
    )

    # Remove proteins with only one peptide if requested
    if remove_single_peptide:
        single_peptide_proteins = pep_counts[pep_counts == 1].index
        # Optionally save these proteins to CSV
        if single_peptide_csv is not None:
            # Get the indices/rows for these proteins
            singles = df.loc[df.index.get_level_values('Accession').isin(single_peptide_proteins)]
            singles.to_csv(single_peptide_csv)
            print(f"Single-peptide proteins ({len(single_peptide_proteins)}) exported to {single_peptide_csv}")
        # Remove them from the protein matrix
        aggdf = aggdf.loc[~aggdf.index.isin(single_peptide_proteins)]
        print(f"Filtered out {len(single_peptide_proteins)} single-peptide proteins.")

    return aggdf
    
# ----------------------- Method 3: Tukey's Median Polish (iterative) ----------------------------------
def aggregate_median_polish(df, max_iter=MAX_ITER, tol=TOL, verbose=False):
    # Prepare outputs
    proteins = df.index.get_level_values(0).unique()
    samples = df.columns
    results = pd.DataFrame(index=proteins, columns=samples, dtype=float)

    for protein in proteins:
        sub = df.xs(protein, level=0)
        mat = sub.values.astype(float)

        # Overall effect on original scale
        overall = np.nanmedian(mat)

        mask = ~np.isnan(mat)
        n_rows, n_cols = mat.shape
        row_effect = np.zeros(n_rows)
        col_effect = np.zeros(n_cols)

        # Iterative median polish on residuals
        # (you can optionally start from mat - overall, but it's not required
        #  if you never again use 'overall' inside the loop)
        for it in range(max_iter):
            row_medians = np.nanmedian(mat, axis=1)
            row_effect += row_medians
            for i in range(n_rows):
                mat[i, mask[i, :]] -= row_medians[i]

            col_medians = np.nanmedian(mat, axis=0)
            col_effect += col_medians
            for j in range(n_cols):
                mat[mask[:, j], j] -= col_medians[j]

            max_change = max(np.nanmax(np.abs(row_medians)),
                             np.nanmax(np.abs(col_medians)))
            if verbose and it == 0 and len(proteins) < 10:
                print(f"Protein {protein}, iteration {it+1}, max change: {max_change:.3g}")
            if max_change < tol:
                break

        # mat is now (approx) residuals with median ~0, so don't re-use its median.
        # Final protein quantification on original scale:
        results.loc[protein] = overall + col_effect

    return results
    
# ----------------------- Method 3: Identify single-peptide proteins ----------------------------------
def get_single_peptide_proteins(peptide_df_filtered, protein_df, output_csv=OUTPUT_SINGLE_PEPTIDE):
    try:
        peptide_to_protein = peptide_df_filtered.index.get_level_values('Accession')
    except Exception:
        raise ValueError("Expected MultiIndex with a level named 'Accession' on peptide_df_filtered.index")
    from collections import Counter
    peptide_counts = Counter(peptide_to_protein)
    # List proteins that have only ONE peptide after all filtering
    one_peptide_proteins = [protein for protein, count in peptide_counts.items() if count == 1]
    # Make a DataFrame listing all such proteins and their one peptide's info
    idx = peptide_df_filtered.index
    # get the rows where protein is in the single-peptide set
    single_rows = [i for i, acc in enumerate(peptide_to_protein) if acc in one_peptide_proteins]
    # Create final DataFrame listing protein, peptide, and optional annotation
    single_peptide_df = peptide_df_filtered.iloc[single_rows].copy()
    # Add a column for which protein this came from (sometimes redundant if in MultiIndex)
    if 'Accession' not in single_peptide_df.columns:
        single_peptide_df['Accession'] = single_peptide_df.index.get_level_values('Accession')
    # Save out
    single_peptide_df.to_csv(output_csv)
    print(f"Proteins aggregated from single peptide hits: {len(one_peptide_proteins)}")
    return one_peptide_proteins, single_peptide_df

# START

In [ ]:
# 0. Load data
df = pd.read_csv(INPUT_PATH)
meta_df = pd.read_csv(METADATA_PATH)
sample_ids = meta_df['sample_id'].values
sample_labels = meta_df['label'].values

In [ ]:
# 1. Wrangle dfs and check for outliers
peptide_df = df.copy()
prefixes = ['Accession', 'Peptide', 'Area']
selected_columns2 = df.columns[[any(prefix in col for prefix in prefixes) for col in df.columns]] # select sample id columns and accession, peptide
peptide_df = peptide_df.loc[:, selected_columns2]
peptide_df = peptide_df.dropna(subset=['Accession']) # remove NaN
peptide_df = peptide_df[~peptide_df['Accession'].str.contains('#CONTAM#')] # Remove contaminants
peptide_df['Accession'] = peptide_df['Accession'].str.split('|').str[0] # Split Accession string to aquire Uniprot Accession ID
peptide_df.set_index(['Accession', 'Peptide'], inplace=True)
peptide_df = peptide_df.iloc[:, :-(N_GROUP)]
peptide_df = peptide_df[sample_ids]  # Ensure correct column order

group_map = meta_df.set_index('sample_id')['group'].to_dict()
grouped_samples = meta_df.groupby('group')['sample_id'].apply(list).to_dict() # Group metadata

log_data = np.log(peptide_df + 0.00001) # RLE 
peptide_df = peptide_df.replace(0, np.nan)
dfboxen = log_data.T.apply(lambda x: x-np.nanmedian(x))

# Plot
fig = sns.boxenplot(
    data=dfboxen.iloc[:,1:].T,
    orient='h',
    width_method="linear")
plt.xlabel('$Log10$ Relative Abundance')
plt.ylabel('Sample')
fig.set_yticks(sample_ids)
fig.set_yticklabels(sample_labels)
plt.title('RLE $Pre$-Normalization')
plt.savefig(OUTPUT_RLE)
plt.show(fig)

In [ ]:
# 2. Filter for completeness
def intragroup_completeness(row):
    for group, samples in grouped_samples.items():
        completeness = row[samples].count() / len(samples)
        if completeness > MIN_COMPLETENESS:
            return True
    return False

mask_complete = peptide_df.apply(intragroup_completeness, axis=1)
peptide_df = peptide_df[mask_complete]

# 3. Log2 transform
peptide_df_log = np.log2(peptide_df + PSEUDOCOUNT)

# Step 4: QRILC-style impute
log2_imputed = left_tail_skewnorm_impute(peptide_df_log, quantile=QUANTILE, seed=SEED) # Adjust as neccessary

plt.figure(figsize=(8, 5))
plt.hist(log2_imputed.values.flatten(), bins=80, color='slateblue')
plt.xlabel("Log2 Intensity")
plt.ylabel("Frequency")
plt.title("Protein Intensities After $Skewnorm$ Left-Tail Imputation")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_IMPUTE)
plt.show()

In [ ]:
# Dimension Reduction of Prenormalized data with PCA
X0 = log2_imputed.T
pca = PCA(n_components=min(X0.shape))
components = pca.fit_transform(X0)

fig = px.scatter(pd.DataFrame(components).rename(columns={0: 'PC 1', 1: 'PC 2'}), x='PC 1', y='PC 2',
                 color=meta_df['group'],
                 template='plotly_white',
                 labels={
                 'PC 1': f'PC 1: {round(pca.explained_variance_ratio_[0] * 100, 1)}%',
                 'PC 2': f'PC 2: {round(pca.explained_variance_ratio_[1] * 100, 1)}%',
                 'color': LEGEND_LAB},
                 text=sample_labels)

fig.data[0].name = LEGEND_LAB1
fig.data[1].name = LEGEND_LAB2

fig.update_layout(font=dict(family='Arial', size=14), legend=dict(
    orientation='h',
    yanchor='bottom',
    y=1.01,
    xanchor='center',
    x=0.5,
    font=dict(family='Arial', size=14)
))

fig.update_traces(textposition='top center')
fig.update_traces(marker_size=10)
fig.write_image(OUTPUT_PCA)
fig.show()

In [ ]:
# 5. Median Normalization
peptide_df_norm = median_normalize(log2_imputed)

# 6. Filter by peptide intensity
medians = peptide_df_norm.median(axis=1)
intensity_threshold = medians.quantile(MIN_INTENSITY_Q)
mask_intensity = medians >= intensity_threshold
peptide_df_norm = peptide_df_norm[mask_intensity]

# 7. Filter by intragroup CV
def intragroup_cv_ok(row):
    for group, samples in grouped_samples.items():
        values = row[samples]
        if values.mean() == 0:
            return False
        cv = values.std() / values.mean()
        if cv > CV_THRESHOLD or np.isnan(cv):
            return False
    return True

mask_cv = peptide_df_norm.apply(intragroup_cv_ok, axis=1)
peptide_df_filtered = peptide_df_norm[mask_cv]
peptide_df_filtered.to_csv(OUTPUT_FILTERED)

# 8. Aggregate by Tukey's Median Polish
protein_df = aggregate_median_polish(peptide_df_filtered)

# Export for limma and report proteins aggregated/peptides retained
single_peptide_proteins, single_peptide_df = get_single_peptide_proteins(peptide_df_filtered, protein_df, output_csv=OUTPUT_SINGLE_PEPTIDE)
unambiguous_proteins = protein_df[~protein_df.index.str.contains(';')]
final_protein_df = unambiguous_proteins[~unambiguous_proteins.index.isin(single_peptide_proteins)]
final_protein_df.to_csv(OUTPUT_AGGREGATED)
print(f"Retained peptides: {peptide_df_filtered.shape[0]}")
print(f"Aggregated proteins: {protein_df.shape[0]}")

In [ ]:
# Dimension Reduction
X = protein_df.T
y = meta_df.group

umap_2d = UMAP(n_components=2, n_neighbors=X.shape[0]//N_GROUP, random_state=42)
umap_3d = UMAP(n_components=3, n_neighbors=X.shape[0]//N_GROUP, random_state=42)

proj_2d = umap_2d.fit_transform(X)
proj_3d = umap_3d.fit_transform(X)

fig_2d = px.scatter(
    pd.DataFrame(proj_2d).rename(columns={0: 'UMAP1', 1: 'UMAP2'}),
    x='UMAP1',
    y='UMAP2',
    color=y,
    template='plotly_white',
    labels={'color': LEGEND_LAB},
    text=sample_labels
)
fig_2d.data[0].name = LEGEND_LAB1
fig_2d.data[1].name = LEGEND_LAB2

fig_2d.update_layout(font=dict(family='Arial', size=14), legend=dict(
    orientation='h',
    yanchor='bottom',
    y=1.01,
    xanchor='center',
    x=0.5,
    font=dict(family='Arial', size=14)
))

fig_2d.update_traces(textposition='top center')
fig_2d.update_traces(marker_size=10)

fig_3d = px.scatter_3d(
    pd.DataFrame(proj_3d).rename(columns={0: 'UMAP1', 1: 'UMAP2', 2: 'UMAP3'}),
    x='UMAP1', y='UMAP2', z='UMAP3',
    color=y,
    labels={'color': LEGEND_LAB},
    #text=X.index
)

fig_3d.update_traces(textposition='top center')
fig_3d.update_traces(marker_size=6)

fig_2d.write_image(OUTPUT_UMAP2D)
fig_2d.show()
fig_3d.write_image(OUTPUT_UMAP3D)
fig_3d.show()

# Feature Selection

In [ ]:
# Params
DEP_PATH = f'{_PATH_}limma_results-isoforms.csv'
BIOMARKERS_PATH = f'{_PATH_}biomarkers-isoforms.csv'
PROTEIN_DF_PATH = f'{_PATH_}aggregated_protein_matrix.csv'
OUTPUT_KNN_ACCURACY = f'{_PATH_}RFE_knn_accuracy.csv'
OUTPUT_CV_ACCURACY = f'{_PATH_}cv_accuracy.csv'
OUTPUT_RFECV_CURVE = f"{_PATH_}rfe_curve.pdf"
OUTPUT_FM = f"{_PATH_}feature-model_selection.csv"
OUTPUT_ROC = f"{_PATH_}AUROC.pdf"
OUTPUT_PRC = f"{_PATH_}precision-recall_curve.pdf"
OUTPUT_CM = f"{_PATH_}confusion-matrix.pdf"

KNN = KNeighborsClassifier(n_neighbors=2)
SVM= SVC(kernel='linear', probability=True, random_state=SEED)
DT = DecisionTreeClassifier(random_state=SEED)
RF = RandomForestClassifier(random_state=SEED)
LOG = LogisticRegression(max_iter=5000, random_state=SEED)
#VCLF = VotingClassifier()
MODELS = [SVM, DT, RF, LOG]

NUM_FEATURES = 10       # Number of features retained in RFE
CV = 5                  # Number of CV splits
STEP = 1                # Number of features to eliminate after each iteration

## Feature Selection Method

In [ ]:
# Recursive Feature Elimination (RFE)
def RFE_fs(X, y, num_features=NUM_FEATURES):
    X_rfe = {}
    for model in MODELS:
        selector = RFE(estimator=model, n_features_to_select=num_features)
        selector.fit(X, y)
        X_selected = selector.transform(X)
        selected_features = pd.Series(selector.support_, index=X.columns)
        print(f'RFE {model} selected features:')
        print(selected_features[selected_features == True].index.tolist())
        X_rfe[f'{model}'] = (selected_features[selected_features == True].index.tolist())
    return X_rfe

# Recursive Feature Elimination with CV
def RFECV_fs(X, y, MODELS, cv=CV, scoring='roc_auc', step=STEP):
    X_rfecv = {}
    for model in MODELS:
        print(f'\n---- RFECV with model: {model.__class__.__name__} ----')
        # Use StratifiedKFold for classification
        cv_strategy = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
        selector = RFECV(
            estimator=model,
            step=step,
            cv=cv_strategy,
            scoring=scoring,
            n_jobs=-1  # use multicore if possible
        )
        selector.fit(X, y)
        selected = pd.Series(selector.support_, index=X.columns)
        selected_features = selected[selected == True].index.tolist()
        print(f'Optimal number of features: {selector.n_features_}')
        print(f'Selected features: {selected_features}')
        # Store results
        X_rfecv[model.__class__.__name__] = {
            'features': selected_features,
            'rfecv_support': selector.support_,
            'rfecv_mean_test_scores': selector.cv_results_['mean_test_score'],
            'n_features': selector.n_features_
        }
    return X_rfecv

# Evaluate KNN Classifier
def evaluate_knn(X, y, X_rfecv, cv=2):
    scores_rfe = {}
    for model_name, feature_list in X_rfecv.items():
        X_selected = X[feature_list]  # select columns from original DataFrame X
        scores = cross_val_score(KNN, X_selected, y, cv=cv, scoring='roc_auc')
        print(f'\n{model_name}: KNN AUROC = {scores.mean():.4f} ± {scores.std():.4f}')
        scores_rfe[model_name] = scores
    return scores_rfe

## Model Selection Methods

In [ ]:
# Initialize models
models = MODELS

cv = StratifiedKFold(n_splits=CV, shuffle=True, random_state=SEED)

# Cross-Validation Accuracy
def evaluate_cv_accuracy(models, X, y, cv):
    print("Cross-Validated Accuracy:")
    for name, model in models.items():
        scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
        print(f"  {name}: {scores.mean():.4f} ± {scores.std():.4f}")

# Confusion Matrix
def plot_confusion_matrix_cv(model, X, y, cv, name="Model"):
    y_pred = cross_val_predict(model, X, y, cv=cv)
    cm = confusion_matrix(y, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(values_format="d", cmap="Blues")
    plt.title(f"Confusion Matrix: {name}")
    plt.show()

# ROC Curve
def plot_roc_curve_cv(model, X, y, cv, name="Model"):
    try:
        y_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
        fpr, tpr, _ = roc_curve(y, y_proba)
        auc_score = roc_auc_score(y, y_proba)
        plt.figure()
        plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.2f})")
        plt.plot([0, 1], [0, 1], 'k--')
        plt.title(f"ROC Curve: {name}")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.grid()
        plt.legend()
        plt.show()
    except:
        print(f"{name} does not support predict_proba")

# Precision-Recall Curve
def plot_pr_curve_cv(model, X, y, cv, name="Model"):
    try:
        y_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
        precision, recall, _ = precision_recall_curve(y, y_proba)
        avg_prec = average_precision_score(y, y_proba)
        plt.figure()
        plt.plot(recall, precision, label=f"{name} (AP = {avg_prec:.2f})")
        plt.title(f"Precision-Recall Curve: {name}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.grid()
        plt.legend()
        plt.show()
    except:
        print(f"{name} does not support predict_proba")

from sklearn.base import clone

def compare_roc_curves(models, X, y, cv):
    plt.figure(figsize=(8,6))
    any_curve = False
    for name, model in models.items():
        try:
            y_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
            fpr, tpr, _ = roc_curve(y, y_proba)
            auc_score = roc_auc_score(y, y_proba)
            plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.2f})")
            any_curve = True
        except Exception as e:
            print(f"{name} skipped in ROC comparison – {e}")
    if any_curve:
        plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
        plt.title("ROC Curve Comparison")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate (Recall/Sensitivity)")
        plt.legend()
        plt.grid(True)
        plt.savefig(OUTPUT_ROC)
        plt.show()
    else:
        print("No ROC curves to plot.")

def compare_pr_curves(models, X, y, cv):
    plt.figure(figsize=(8,6))
    any_curve = False
    for name, model in models.items():
        try:
            y_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
            precision, recall, _ = precision_recall_curve(y, y_proba)
            avg_prec = average_precision_score(y, y_proba)
            plt.plot(recall, precision, label=f"{name} (AP = {avg_prec:.2f})")
            any_curve = True
        except Exception as e:
            print(f"{name} skipped in PR comparison – {e}")
    if any_curve:
        plt.title("Precision-Recall Curve Comparison")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.legend()
        plt.grid(True)
        plt.savefig(OUTPUT_PRC)
        plt.show()
    else:
        print("No PR curves to plot.")

## Start

In [ ]:
# 0. Load Data
#DEP = pd.read_csv(DEP_PATH)
BIOMARKERS = pd.read_csv(BIOMARKERS_PATH).Accession.to_list()

X1 = protein_df.T
X1 = X1[BIOMARKERS]
y1 = meta_df.set_index('sample_id')
X, y = X1.align(y1, join='inner', axis=0)
y = y['group']
y = np.where(y == 'treatment', 1, 0)
y = np.array(y).ravel()

In [ ]:
# Feature selection WITH CV
X_rfecv = RFECV_fs(X, y, MODELS, cv=CV, scoring='roc_auc', step=STEP)

# Plot RFECV performance curve
for k, v in X_rfecv.items():
    plt.plot(range(1, len(v['rfecv_mean_test_scores'])+1), v['rfecv_mean_test_scores'], label=k)
plt.xlabel('Number of features')
plt.ylabel('Cross-validated accuracy')
plt.legend()
plt.title('RFECV performance curves')
plt.savefig(OUTPUT_RFECV_CURVE)
plt.show()

In [ ]:
# Model selection
# Sample classifier list
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Naive Bayes': GaussianNB(),
    'KNN': KNeighborsClassifier(),
    'LDA': LinearDiscriminantAnalysis(),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Extra Trees': ExtraTreesClassifier(random_state=42),
    'Bagging': BaggingClassifier(random_state=42),
    'Voting Ensemble': VotingClassifier(estimators=[
        ('lr', LogisticRegression(max_iter=1000, random_state=42)),
        ('dt', DecisionTreeClassifier(random_state=42)),
        ('svc', SVC(probability=True, random_state=42))
    ], voting='soft')
}

seeds = range(100)
all_results = []

for sel_model, info in X_rfecv.items():            # Each feature selection
    feats = info['features']
    X1 = X[feats]
    y = y
    for clf_name, clf in classifiers.items():            # Each classifier
        accuracies = []
        aucs = []
        for seed in seeds:
            # Try/except block as before to ensure no AUC errors
            X_train, X_test, y_train, y_test = train_test_split(
                X1, y, test_size=0.3, random_state=seed, stratify=y
            )
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            accuracies.append(acc)
            if hasattr(clf, "predict_proba"):
                y_proba = clf.predict_proba(X_test)[:, 1]
                try:
                    auc = roc_auc_score(y_test, y_proba)
                except ValueError:
                    auc = np.nan
                aucs.append(auc)
            else:
                aucs.append(np.nan)
        all_results.append({
            'features_selected_by': sel_model,
            'classifier': clf_name,
            'mean_accuracy': np.mean(accuracies),
            'std_accuracy': np.std(accuracies),
            'mean_auc': np.nanmean(aucs),
            'std_auc': np.nanstd(aucs)
        })

results_df = pd.DataFrame(all_results)
results_df.to_csv(OUTPUT_FM)

In [ ]:
# Select Best Set
best_set = results_df[[
    'features_selected_by', 'mean_auc', 'mean_accuracy'
]].groupby('features_selected_by').agg('sum').sort_values(by=['mean_auc', 'mean_accuracy'], ascending=False).index[0]

bm = X_rfecv[best_set]['features']

In [ ]:
# Run Evaluations
evaluate_cv_accuracy(classifiers, X[bm], y, cv)

for name, model in classifiers.items():
    print(f'\nEvaluating {name}')
    plot_confusion_matrix_cv(model, X[bm], y, cv, name)
    plot_roc_curve_cv(model, X[bm], y, cv, name)
    plot_pr_curve_cv(model, X[bm], y, cv, name)

print('\nComparing All Models (ROC & PR Curves):')
compare_roc_curves(classifiers, X[bm], y, cv)
compare_pr_curves(classifiers, X[bm], y, cv)
                                   

In [ ]:
# List of features to test
p_list = bm

# final result tracker
all_results = []

# Seed list for evaluation
seeds = range(1000)  # Use 1000 for full-scale testing

# Loop through every single predictor
for name, clf in classifiers.items():
    accuracies = []
    for seed in seeds:
        X_train, X_test, y_train, y_test = train_test_split(X[p_list], y, test_size=0.3, random_state=seed)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        accuracies.append(acc)
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)

    all_results.append({
        'Feature': p_list,
        'Model': name,
        'Mean Accuracy': mean_acc,
        'Std Accuracy': std_acc,
    })
    print(f"{p_list} | {name}: Accuracy = {mean_acc:.4f} ± {std_acc:.4f}")

# Convert to DataFrame
results_df = pd.DataFrame(all_results)

# Find and display the best-combination
best_row = results_df.loc[results_df['Mean Accuracy'].idxmax()]
print("\nBest Feature + Model Combination:")
print(best_row)

# plot confusion matrix... best row
value = classifiers[best_row.iloc[1]]
y_pred = cross_val_predict(value, X[bm], y, cv=cv)
cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(values_format="d", cmap="Blues")
plt.title(f"Confusion Matrix: {best_row.iloc[1]}")
plt.savefig(OUTPUT_CM)
plt.show()
